<a href="https://colab.research.google.com/github/ronykris/capstone-group10/blob/feat-detection/yolo11_foodDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 901.3/901.3 kB 50.2 MB/s eta 0:00:00


In [2]:
import torch
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
!pip install opencv-python pillow tqdm pyyaml numpy

In [4]:
!unzip /content/YoloDatasetUpdated.zip

Streaming output truncated to the last 5000 lines.
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0092.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0177.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0462.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0403.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0249.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0137.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0100.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0269.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0333.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0270.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0195.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0471.jpg  
  inflating: 

**Convert segmentation dataset to yolo formated detection set**

In [ ]:
import os
import numpy as np
from pathlib import Path

def convert_segmentation_to_detection(label_content):
    """Convert segmentation format to detection format"""
    values = label_content.strip().split()
    if len(values) < 5:
        return None

    class_id = values[0]

    # Extract x,y coordinates from segmentation format
    coordinates = [float(x) for x in values[1:]]
    x_coords = coordinates[::2]
    y_coords = coordinates[1::2]

    # Calculate bounding box
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)

    # Convert to YOLO detection format (center_x, center_y, width, height)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min

    # Return YOLO detection format string
    return f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"

def convert_dataset(base_path):
    """Convert entire dataset to detection format"""
    base_path = Path(base_path)

    for split in ['train', 'val', 'test']:
        labels_dir = base_path / split / 'labels'
        if not labels_dir.exists():
            continue

        print(f"Converting {split} labels...")
        for label_file in labels_dir.glob('*.txt'):
            try:
                # Read original content
                with open(label_file, 'r') as f:
                    content = f.read().strip()

                # Convert format
                new_content = convert_segmentation_to_detection(content)

                if new_content:
                    # Save in detection format
                    with open(label_file, 'w') as f:
                        f.write(new_content)

            except Exception as e:
                print(f"Error processing {label_file}: {str(e)}")

    print("Conversion complete!")

# Convert the dataset
dataset_path = '/content/YoloDataset'
convert_dataset(dataset_path)



**Creating synthetic data for multiple food items in one image**

In [ ]:
import cv2
import numpy as np
import random
import os
from pathlib import Path
import glob

def create_and_verify_synthetic_dataset():
    # Set paths
    dataset_path = '/content/YoloDatasetUpdated'
    synthetic_path = '/content/YoloDatasetUpdated/synthetic'

    # Create synthetic images
    print("Creating synthetic images...")
    create_synthetic_images(dataset_path, synthetic_path, num_synthetic=500)

    # Verify creation using glob
    synthetic_images = len(glob.glob(os.path.join(synthetic_path, 'images', '*.jpg')))
    synthetic_labels = len(glob.glob(os.path.join(synthetic_path, 'labels', '*.txt')))

    print(f"\nCreated synthetic dataset:")
    print(f"Images: {synthetic_images}")
    print(f"Labels: {synthetic_labels}")

    return synthetic_path

def resize_with_aspect_ratio(image, target_size=640):
    """Resize image while maintaining aspect ratio"""
    h, w = image.shape[:2]
    scale = min(target_size/w, target_size/h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    return cv2.resize(image, (new_w, new_h))

def create_synthetic_images(dataset_path, output_path, num_synthetic=1000):
    """Create synthetic images with multiple food items"""

    # Convert paths to strings and use glob
    images_path = os.path.join(dataset_path, 'train', 'images')
    labels_path = os.path.join(dataset_path, 'train', 'labels')
    output_images = os.path.join(output_path, 'images')
    output_labels = os.path.join(output_path, 'labels')

    # Create output directories
    os.makedirs(output_images, exist_ok=True)
    os.makedirs(output_labels, exist_ok=True)

    # Get list of image files using glob
    image_files = glob.glob(os.path.join(images_path, '*.jpg'))

    for i in range(num_synthetic):
        # Create blank canvas (white background)
        canvas = np.ones((640, 640, 3), dtype=np.uint8) * 255
        combined_labels = []

        # Randomly select 2-4 images to combine
        num_items = random.randint(2, 4)
        selected_images = random.sample(image_files, num_items)

        for idx, img_path in enumerate(selected_images):
            try:
                # Read image and label
                img = cv2.imread(img_path)
                if img is None:
                    continue

                # Get corresponding label path
                label_path = os.path.join(labels_path,
                                        os.path.splitext(os.path.basename(img_path))[0] + '.txt')
                if not os.path.exists(label_path):
                    continue

                with open(label_path, 'r') as f:
                    label = f.read().strip().split()

                # First resize to maintain aspect ratio
                img_resized = resize_with_aspect_ratio(img, target_size=320)
                new_h, new_w = img_resized.shape[:2]

                # Calculate random position
                x_pos = random.randint(0, max(0, 640 - new_w))
                y_pos = random.randint(0, max(0, 640 - new_h))

                # Create mask for smooth blending
                mask = np.ones(img_resized.shape[:2], dtype=np.float32)
                mask = cv2.GaussianBlur(mask, (7, 7), 0)

                # Place image on canvas with blending
                for c in range(3):
                    canvas[y_pos:y_pos+new_h, x_pos:x_pos+new_w, c] = \
                        canvas[y_pos:y_pos+new_h, x_pos:x_pos+new_w, c] * (1 - mask) + \
                        img_resized[:, :, c] * mask

                # Adjust label coordinates
                class_id = label[0]
                x_center = (x_pos + new_w/2) / 640
                y_center = (y_pos + new_h/2) / 640
                width = new_w / 640
                height = new_h / 640

                # Ensure coordinates are within bounds
                x_center = min(max(x_center, 0), 1)
                y_center = min(max(y_center, 0), 1)
                width = min(width, 1)
                height = min(height, 1)

                combined_labels.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

            except Exception as e:
                print(f"Error processing {img_path}: {str(e)}")
                continue

        if combined_labels:  # Only save if we have valid labels
            # Save synthetic image and labels
            output_img_path = os.path.join(output_images, f"synthetic_{i:04d}.jpg")
            output_label_path = os.path.join(output_labels, f"synthetic_{i:04d}.txt")

            cv2.imwrite(output_img_path, canvas)
            with open(output_label_path, 'w') as f:
                f.write('\n'.join(combined_labels))

            if i % 100 == 0:
                print(f"Created {i} synthetic images")

# Create the synthetic dataset
synthetic_path = create_and_verify_synthetic_dataset()

**Train combined dataset**

In [5]:
from ultralytics import YOLO

def train_combined_dataset():
    # Start with pretrained model
    model = YOLO('yolo11n.pt')

    # Train with combined data
    results = model.train(
        data='/content/YoloDatasetUpdated/data.yaml',
        #data='/content/FoodObjectDetectionYolov11/data.yaml',
        epochs=10,
        imgsz=640,
        batch=16,
        patience=20,
        lr0=0.01,
        # Multiple object detection parameters
        mosaic=1.0,
        mixup=0.5,
        copy_paste=0.5,
        max_det=20
    )
    return model

model = train_combined_dataset()

100%|██████████| 5.35M/5.35M [00:00<00:00, 233MB/s]


Ultralytics 8.3.51 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/content/YoloDatasetUpdated/data.yaml, epochs=10, time=None, patience=20, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=20, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, s

100%|██████████| 755k/755k [00:00<00:00, 141MB/s]



                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128

train: Scanning /content/YoloDatasetUpdated/synthetic/labels.cache... 4137 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4137/4137 [00:00<?, ?it/s]


...

      10/10      2.61G     0.3312      1.868      1.088         16        640: 100%|██████████| 259/259 [01:28<00:00,  2.93it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 46/46 [00:13<00:00,  3.29it/s]

                   all       1446       1446      0.499       0.51      0.523       0.48



10 epochs completed in 0.294 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 5.5MB
Optimizer stripped from runs/detect/train/weights/best.pt, 5.5MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.51 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
YOLO11n summary (fused): 238 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 46/46 [00:15<00:00,  3.06it/s]


                   all       1446       1446      0.499      0.512      0.523       0.48
             adhirasam         18         18      0.659      0.778      0.763      0.683
             aloo_gobi         19         19      0.691      0.684      0.677      0.625
            aloo_matar         18         18      0.479      0.167       0.28      0.254
            aloo_methi         19         19      0.326          1      0.844      0.814
     aloo_shimla_mirch         18         18      0.328      0.333      0.256      0.251
            aloo_tikki         19         19      0.685      0.368      0.493      0.373
                anarsa         19         19      0.364      0.526      0.516      0.454
               ariselu         18         18      0.377      0.833      0.592      0.424
          bandar_laddu         20         20      0.516      0.799      0.777      0.771
               basundi         18         18      0.303      0.333      0.307       0.26
               bhatur

**Inference the model**

In [7]:
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image

def run_inference():
    # Load your trained model
    model = YOLO('/content/runs/detect/train/weights/best.pt')  # adjust path as needed

    # Run inference on an image
    results = model.predict(
        source='/content/YoloDatasetUpdated/val/images/002c57eb72.jpg',  # can be image/folder/video
        save=True,                # save results
        conf=0.1,               # confidence threshold
        iou=0.45,
        save_txt=True,           # save results in txt file
        save_crop=True           # save cropped predictions
    )

    # Process results
    for result in results:
        # Get masks
        if result.masks is not None:
            masks = result.masks.data.cpu().numpy()

        # Get boxes
        if result.boxes is not None:
            boxes = result.boxes.data.cpu().numpy()

        # Get class names for predictions
        class_names = [model.names[int(class_id)] for class_id in result.boxes.cls]

        print(f"Found {len(class_names)} objects: {class_names}")

# Run on a single image
def predict_image(model_path, image_path):
    # Load model
    model = YOLO(model_path)

    # Run inference
    results = model.predict(
        source=image_path,
        save=True,
        conf=0.25
    )

    print(f"Results saved to {results[0].save_dir}")
    return results

# Run on multiple images in a folder
def predict_folder(model_path, folder_path):
    model = YOLO(model_path)
    results = model.predict(folder_path, save=True, conf=0.25)
    return results

# Example usage:
MODEL_PATH = '/content/runs/detect/train/weights/best.pt'
IMAGE_PATH = '/content/YoloDatasetUpdated/val/images/002c57eb72.jpg'

# For single image
#results = predict_image(MODEL_PATH, IMAGE_PATH)
run_inference()

# For folder of images
#FOLDER_PATH = 'path/to/test/folder'
#results = predict_folder(MODEL_PATH, FOLDER_PATH)


image 1/1 /content/YoloDatasetUpdated/val/images/002c57eb72.jpg: 384x640 1 mysore_pak, 50.6ms
Speed: 1.7ms preprocess, 50.6ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
Results saved to runs/detect/predict
1 label saved to runs/detect/predict/labels
Found 1 objects: ['mysore_pak']


**Validate model**

In [6]:
from ultralytics import YOLO

# Load model
model = YOLO('/content/runs/detect/train/weights/best.pt')

# Run model validation on the validation set
metrics = model.val(data='/content/YoloDatasetUpdated/data.yaml')
print("Validation Metrics:", metrics)

Ultralytics 8.3.51 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
YOLO11n summary (fused): 238 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs


val: Scanning /content/YoloDatasetUpdated/val/labels.cache... 1446 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1446/1446 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 91/91 [00:17<00:00,  5.19it/s]


                   all       1446       1446      0.494      0.516      0.523      0.479
             adhirasam         18         18      0.663      0.722      0.743      0.659
             aloo_gobi         19         19      0.723      0.684      0.699      0.638
            aloo_matar         18         18      0.462      0.167       0.28      0.251
            aloo_methi         19         19      0.338          1      0.848      0.817
     aloo_shimla_mirch         18         18      0.357      0.372      0.249      0.244
            aloo_tikki         19         19      0.708      0.368        0.5      0.376
                anarsa         19         19      0.355      0.526      0.507       0.44
               ariselu         18         18      0.353      0.833      0.582      0.416
          bandar_laddu         20         20      0.515        0.8      0.782      0.775
               basundi         18         18      0.329      0.355      0.308       0.26
               bhatur

In [9]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"
!zip -r YoloDatasetUpdatedRuns.zip /content/runs

  adding: content/runs/ (stored 0%)
  adding: content/runs/detect/ (stored 0%)
  adding: content/runs/detect/predict/ (stored 0%)
  adding: content/runs/detect/predict/labels/ (stored 0%)
  adding: content/runs/detect/predict/labels/002c57eb72.txt (deflated 13%)
  adding: content/runs/detect/predict/crops/ (stored 0%)
  adding: content/runs/detect/predict/crops/mysore_pak/ (stored 0%)
  adding: content/runs/detect/predict/crops/mysore_pak/002c57eb72.jpg (deflated 3%)
  adding: content/runs/detect/predict/002c57eb72.jpg (deflated 4%)
  adding: content/runs/detect/val/ (stored 0%)
  adding: content/runs/detect/val/val_batch0_pred.jpg (deflated 10%)
  adding: content/runs/detect/val/val_batch2_labels.jpg (deflated 8%)
  adding: content/runs/detect/val/val_batch1_labels.jpg (deflated 7%)
  adding: content/runs/detect/val/R_curve.png (deflated 9%)
  adding: content/runs/detect/val/F1_curve.png (deflated 9%)
  adding: content/runs/detect/val/PR_curve.png (deflated 7%)
  adding: content/runs/